# Assignment 1A — Medical & Clinical Literature

| S. No | Name | Bits ID |
|-------|------|---------|
| 1 | Madhu Gururajan | 2025cs05027 |
| 2 | Anand Bharti | 2025cs05036 |

One notebook, **Runtime → Run all**. There are no `RUN_*` switches.

**Part A (10 marks)** — PDF corpus → clean ≥10 MB → pack tokens → inspect BioGPT-Large → continual pre-training (CPT) → domain perplexity and a forgetting check.

**Part B (5 marks)** — ≥100 instruction pairs → three QLoRA adapters (A/B/C) → compare them to the Part A baseline.

**Hardware:** `src/runtime.py` picks the recipe from the machine. A Colab T4 (~15 GB) runs about **60 CPT steps** (~1 hour at the measured ~57 s/step) with 8-bit Adam and 4-bit QLoRA. A faster lab GPU (A100/L40S) runs about **one packed epoch**. A **CPU (or Apple MPS) runtime still trains**: it cannot hold full AdamW on 1.5B weights, so CPT uses LoRA, sequence length 256, and a handful of steps so Run all finishes. There are no `RUN_*` switches.


In [31]:
# Colab's "Open notebook from GitHub" copies only this .ipynb, not src/.
# Clone the repo into /content so later cells can `from src.runtime import ...`.
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/madhu-GG/bits_llm_assignment_1_sem3.git"
REPO_NAME = "bits_llm_assignment_1_sem3"

def _has_src(root: Path) -> bool:
    return (root / "src" / "runtime.py").exists() and (root / "src" / "corpus.py").exists()

if _has_src(Path.cwd()):
    print("Already in the assignment folder:", Path.cwd())
elif "google.colab" in sys.modules:
    dest = Path("/content") / REPO_NAME
    if not _has_src(dest):
        print("Cloning", REPO_URL, "into", dest)
        env = os.environ.copy()
        env["GIT_TERMINAL_PROMPT"] = "0"
        clone_url = REPO_URL
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")  # optional Colab secret; never hardcode a token
        except Exception:
            token = None
        if token:
            clone_url = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        subprocess.run(["git", "clone", "--depth", "1", clone_url, str(dest)], check=True, env=env)
        if token:
            subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", REPO_URL], check=True)
    os.chdir(dest)
    print("Working directory:", Path.cwd())
else:
    raise FileNotFoundError(
        "src/ is missing. Open this notebook from the assignment folder, or clone the GitHub repo first."
    )

assert _has_src(Path.cwd()), f"src/ still missing under {Path.cwd()}"


Already in the assignment folder: /content/bits_llm_assignment_1_sem3


In [32]:
# Install once on Colab. Locally, use: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip -q install pypdf pyarrow matplotlib transformers accelerate peft trl datasets bitsandbytes sentencepiece sacremoses psutil huggingface_hub


In [33]:
# Memory env vars must be set BEFORE torch is imported.
from __future__ import annotations

import gc
import json
import hashlib
import logging
import math
import os
import random
import re
import sys
import warnings
import importlib
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

ROOT = Path.cwd()
print(f"We are running in {ROOT}")

# Invalidate and clean Python's path and finder caches to detect newly added directories
sys.path_importer_cache.clear()
importlib.invalidate_caches()

# Clean up any direct 'src' paths and place the root directory at position 0
abs_root = str(ROOT.resolve())
if abs_root not in sys.path:
    sys.path.insert(0, abs_root)

for path_str in list(sys.path):
    if path_str.endswith("/src") or path_str.endswith("\\src"):
        sys.path.remove(path_str)

from src.runtime import (
    apply_memory_env,
    cpt_steps_for_dataset,
    detect_runtime,
    disable_optional_scipy_backend,
    load_causal_lm,
    memory_snapshot,
    merge_lora_if_needed,
    model_audit,
    project_root,
    release_model_memory,
    resolve_lora_targets,
    seed_everything,
    trainer_memory_kwargs,
    training_budget,
    wrap_lora_for_cpt,
)
from src.corpus import MIN_CLEAN_BYTES, ensure_pdf_corpus, iter_source_pdfs

apply_memory_env()
seed_everything(42)

ROOT = project_root()
abs_root = str(ROOT.resolve())
if abs_root not in sys.path:
    sys.path.insert(0, abs_root)

RAW_PDFS = ROOT / "raw_pdfs"
EXTRACTED = ROOT / "extracted_text"
CLEAN_CORPUS = ROOT / "clean_corpus"
OUTPUTS = ROOT / "outputs"
for folder in (EXTRACTED, CLEAN_CORPUS, OUTPUTS):
    folder.mkdir(parents=True, exist_ok=True)

MODEL_ID = "microsoft/biogpt-large"
# Pin the revision that ships safetensors (avoids the legacy .bin load path).
MODEL_REVISION = "4585354702bf941a07727e316182421fea0051fe"
CPT_DIR = OUTPUTS / "biogpt-large-cpt"
PACKED_PARQUET = OUTPUTS / "medical_packed_train.parquet"
ADAPTER_ROOT = OUTPUTS / "qlora_adapters"

DOMAIN_PROMPTS = [
    "Explain why repeated blood-pressure measurements are useful in hypertension.",
    "What is the relationship between insulin resistance and type 2 diabetes?",
    "Why is antimicrobial stewardship important?",
]
GENERAL_PROMPTS = [
    "The capital of France is",
    "Water boils at",
    "The speed of light is approximately",
]
EXPECTED_TERMS = [["Paris"], ["100", "one hundred"], ["299", "300,000", "three hundred"]]

RUNTIME = detect_runtime()
BUDGET = training_budget(RUNTIME)
# 1024 on CUDA; 256 (or 128) on CPU so activations fit next to 6 GB fp32 weights.
SEQUENCE_LENGTH = int(BUDGET.get("sequence_length") or 1024)
print({"root": str(ROOT), "model": MODEL_ID, "runtime": RUNTIME, "budget": BUDGET, "sequence_length": SEQUENCE_LENGTH})
memory_snapshot("startup")

We are running in /content/bits_llm_assignment_1_sem3
{'root': '/content/bits_llm_assignment_1_sem3', 'model': 'microsoft/biogpt-large', 'runtime': {'device': 'cuda', 'dtype': 'bfloat16', 'cuda': True, 'mps': False, 'gpu_name': 'Tesla T4', 'gpu_total_gb': 14.56, 'gpu_free_gb': 7.97, 'cpu_total_gb': 12.67, 'cpu_available_gb': 6.95, 'colab': True, 'gpu_cc': '7.5'}, 'budget': {'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8, 'cpt_max_steps': 300, 'sft_max_steps': 60, 'dpo_epochs': 3, 'four_bit': True, 'eight_bit_adam': True, 'sequence_length': 1024, 'max_new_tokens': 40, 'warmup_ratio': 0.1, 'max_memory': {0: '12GiB', 'cpu': '4GiB'}, 'reason': 'Colab T4-class box (~15 GB GPU / ~15 GB RAM): 8-bit Adam, 4-bit QLoRA, load straight to GPU, one model at a time'}}
{'tag': 'startup', 'cpu_total_gb': 12.67, 'cpu_available_gb': 6.95, 'cpu_process_gb': 5.44, 'gpu_allocated_gb': 5.13, 'gpu_reserved_gb': 6.46, 'gpu_free_gb': 7.97, 'gpu_total_gb': 14.56}


{'tag': 'startup',
 'cpu_total_gb': 12.67,
 'cpu_available_gb': 6.95,
 'cpu_process_gb': 5.44,
 'gpu_allocated_gb': 5.13,
 'gpu_reserved_gb': 6.46,
 'gpu_free_gb': 7.97,
 'gpu_total_gb': 14.56}

## Part A, Step 1 — Collect, extract, and clean PDFs (2 marks)

PDFs come from the public Hugging Face dataset `madhugg93/MedicalText` (`raw_pdfs.tgz`). The 12 `medical_topic_*.pdf` files are one-page starter notes (~0.04 MB) and are ignored. Cleaning order is the one the brief asks for: length → within-document repetition → exact-document dedup → English.


In [34]:
class PdfWarningCollector(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.WARNING)
        self.messages = {}

    def emit(self, record):
        message = record.getMessage()
        normalized = re.sub(r" at byte 0x[0-9a-f]+", " at byte <offset>", message)
        self.messages[normalized] = self.messages.get(normalized, 0) + 1


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_pdfs_page_by_page(pdf_dir, extracted_dir):
    from pypdf import PdfReader
    collector = PdfWarningCollector()
    logger = logging.getLogger("pypdf")
    logger.addHandler(collector)
    logger.setLevel(logging.WARNING)
    logger.propagate = False
    extracted_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    pdfs = iter_source_pdfs(pdf_dir)
    for i, pdf_path in enumerate(pdfs, start=1):
        text_path = extracted_dir / f"{pdf_path.stem}.txt"

        # Skip extraction if the text file already exists to handle interrupted runs
        if text_path.exists():
            try:
                existing_text = text_path.read_text(encoding="utf-8", errors="ignore")
                # Simple heuristic to count pages using our markers if present
                page_count = max(1, len(re.findall(r"\[PAGE \d+\]", existing_text)))
                rows.append({"file": pdf_path.name, "status": "success", "pages": page_count, "characters": len(existing_text)})
                if i % 20 == 0 or i == len(pdfs):
                    print(f"  skipped {i}/{len(pdfs)} PDFs (already extracted)")
                continue
            except Exception:
                pass

        reader = None
        try:
            reader = PdfReader(str(pdf_path), strict=False)
            pages = []
            for n, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").replace("\x00", "").strip()
                pages.append(f"[PAGE {n}]\n{text}")
            extracted = "\n\n".join(pages)
            text_path.write_text(extracted, encoding="utf-8")
            rows.append({"file": pdf_path.name, "status": "success", "pages": len(pages), "characters": len(extracted)})
        except Exception as exc:
            text_path.unlink(missing_ok=True)
            rows.append({"file": pdf_path.name, "status": "error", "pages": 0, "characters": 0, "error": f"{type(exc).__name__}: {exc}"})
        reader = None
        if i % 20 == 0 or i == len(pdfs):
            print(f"  extracted {i}/{len(pdfs)} PDFs")
        if i % 25 == 0:
            gc.collect()
    return rows


def paragraphs(text):
    blocks = re.split(r"\n\s*\n+", text)
    return [re.sub(r"\s+", " ", block).strip() for block in blocks if block.strip()]


def looks_english(text):
    words = re.findall(r"[A-Za-z]+", text.lower())
    stopwords = {"the", "and", "of", "to", "in", "is", "for", "with", "on", "as", "by", "this", "an", "or", "from", "are", "be", "can", "which", "when"}
    if not words:
        return False
    ascii_ratio = sum(ord(ch) < 128 for ch in text) / max(len(text), 1)
    stopword_hits = sum(word in stopwords for word in words)
    return ascii_ratio >= 0.85 and stopword_hits >= max(2, min(10, len(words) // 30))


def clean_extracted_text(extracted_dir, cleaned_dir, min_chars=50, duplicate_fraction=0.30):
    files = sorted(p for p in extracted_dir.glob("*.txt") if not p.name.startswith("medical_topic_"))
    counts = {"before": len(files), "after_length": 0, "after_repetition": 0, "after_deduplication": 0, "after_language": 0}
    candidates = []
    for path in files:
        text = path.read_text(encoding="utf-8", errors="ignore").strip()
        if len(text) >= min_chars:
            candidates.append((path, text))
    counts["after_length"] = len(candidates)
    non_repetitive = []
    for path, text in candidates:
        paras = paragraphs(text)
        duplicate_ratio = (len(paras) - len(set(paras))) / max(len(paras), 1)
        if duplicate_ratio <= duplicate_fraction:
            non_repetitive.append((path, text))
    counts["after_repetition"] = len(non_repetitive)
    seen, unique = set(), []
    for path, text in non_repetitive:
        key = re.sub(r"\s+", " ", text).strip()
        if key not in seen:
            seen.add(key)
            unique.append((path, text))
    counts["after_deduplication"] = len(unique)
    final = [(path, text) for path, text in unique if looks_english(text)]
    counts["after_language"] = len(final)
    cleaned_dir.mkdir(parents=True, exist_ok=True)
    for old in cleaned_dir.glob("*.txt"):
        old.unlink()
    for path, text in final:
        (cleaned_dir / path.name).write_text(text, encoding="utf-8")
    impact = {
        "length_filter": counts["before"] - counts["after_length"],
        "repetition_filter": counts["after_length" ] - counts["after_repetition"],
        "deduplication": counts["after_repetition"] - counts["after_deduplication"],
        "language_filter": counts["after_deduplication"] - counts["after_language"],
    }
    greatest = max(impact, key=impact.get) if max(impact.values(), default=0) > 0 else "none (no documents removed)"
    return counts, impact, greatest


print("Collecting PDFs (local tarball, else Hugging Face madhugg93/MedicalText)...")
corpus_report = ensure_pdf_corpus(ROOT, RAW_PDFS)
source_pdfs = iter_source_pdfs(RAW_PDFS)
if len(source_pdfs) < 100:
    raise FileNotFoundError(
        f"Only {len(source_pdfs)} real PMC papers in {RAW_PDFS}. "
        "Need ~140 from madhugg93/MedicalText, not the 12 starter notes."
    )

extraction_stats = extract_pdfs_page_by_page(RAW_PDFS, EXTRACTED)
clean_counts, clean_impact, greatest_impact = clean_extracted_text(EXTRACTED, CLEAN_CORPUS)
corpus_bytes = sum(p.stat().st_size for p in CLEAN_CORPUS.glob("*.txt"))
step1 = {
    "pdf_documents": len(extraction_stats),
    "pages_extracted": sum(row["pages"] for row in extraction_stats),
    "pdfs_failed": sum(1 for row in extraction_stats if row["status"] == "error"),
    "clean_corpus_bytes": corpus_bytes,
    "clean_corpus_megabytes": round(corpus_bytes / 1024**2, 4),
    "cleaning_counts": clean_counts,
    "removed_by_step": clean_impact,
    "greatest_impact": greatest_impact,
}
if corpus_bytes < MIN_CLEAN_BYTES:
    warnings.warn("Clean corpus is still below 10 MB. CPT will not move perplexity on starter notes.")
print(json.dumps(step1, indent=2))
(OUTPUTS / "step1_cleaning_report.json").write_text(json.dumps(step1, indent=2), encoding="utf-8")
memory_snapshot("after_clean")

{
  "dataset": "madhugg93/MedicalText",
  "archive": "raw_pdfs.tgz",
  "unpack": {
    "tgz": false,
    "zip": false,
    "extracted_members": 0,
    "papers_after_unpack": 140
  },
  "download": {},
  "paper_count": 140,
  "starter_notes_ignored": 12
}
Ready: 140 real PMC papers in /content/bits_llm_assignment_1_sem3/raw_pdfs
  skipped 20/140 PDFs (already extracted)
  skipped 40/140 PDFs (already extracted)
  skipped 60/140 PDFs (already extracted)
  skipped 80/140 PDFs (already extracted)
  skipped 100/140 PDFs (already extracted)
  skipped 120/140 PDFs (already extracted)
  skipped 140/140 PDFs (already extracted)
{
  "pdf_documents": 140,
  "pages_extracted": 2261,
  "pdfs_failed": 3,
  "clean_corpus_bytes": 11235484,
  "clean_corpus_megabytes": 10.715,
  "cleaning_counts": {
    "before": 137,
    "after_length": 137,
    "after_repetition": 137,
    "after_deduplication": 137,
    "after_language": 137
  },
  "removed_by_step": {
    "length_filter": 0,
    "repetition_filter":

{'tag': 'after_clean',
 'cpu_total_gb': 12.67,
 'cpu_available_gb': 8.71,
 'cpu_process_gb': 1.82,
 'gpu_allocated_gb': 3.3,
 'gpu_reserved_gb': 6.46,
 'gpu_free_gb': 7.97,
 'gpu_total_gb': 14.56}

## Held-out split

10% of **documents** are held out for perplexity. The split is by file, not by packed chunk, so evaluation text is never seen during CPT.


In [35]:
def split_corpus_files(cleaned_dir, eval_fraction=0.10, seed=42):
    files = sorted(cleaned_dir.glob("*.txt"))
    rng = random.Random(seed)
    order = files.copy()
    rng.shuffle(order)
    n_eval = max(1, int(round(len(order) * eval_fraction)))
    eval_files = sorted(order[:n_eval])
    train_files = sorted(order[n_eval:])
    manifest = {
        "train_documents": len(train_files),
        "eval_documents": len(eval_files),
        "train_files": [p.name for p in train_files],
        "eval_files": [p.name for p in eval_files],
    }
    (OUTPUTS / "corpus_split.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print({"train_documents": manifest["train_documents"], "eval_documents": manifest["eval_documents"]})
    return train_files, eval_files

train_files, eval_files = split_corpus_files(CLEAN_CORPUS)
eval_texts = [p.read_text(encoding="utf-8") for p in eval_files]


{'train_documents': 123, 'eval_documents': 14}


## Part A, Step 2 — Tokenize and pack (2 marks)

We reuse BioGPT's tokenizer (no custom vocab). Each document is wrapped with BOS/EOS, then all token ids are concatenated and sliced into packed blocks (`SEQUENCE_LENGTH` from the runtime budget: 1024 on CUDA, 256 on a typical CPU) so training is never padded with zeros.


In [36]:
def encode_without_model_limit(tokenizer, text, chunk_chars=4000):
    ids = []
    for start in range(0, len(text), chunk_chars):
        ids.extend(tokenizer.encode(text[start:start + chunk_chars], add_special_tokens=False))
    return ids


def tokenize_and_pack(files, parquet_path, model_id, sequence_length):
    from transformers import AutoTokenizer
    import pyarrow as pa
    import pyarrow.parquet as pq

    disable_optional_scipy_backend()
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=MODEL_REVISION)
    bos_id, eos_id = tokenizer.bos_token_id, tokenizer.eos_token_id
    if bos_id is None:
        bos_id = eos_id if eos_id is not None else tokenizer.unk_token_id
        warnings.warn("Tokenizer has no BOS token; using EOS/UNK.")
    if eos_id is None:
        eos_id = bos_id
        warnings.warn("Tokenizer has no EOS token; using BOS.")

    stream, document_lengths, packed_ids, packed_mask = [], [], [], []
    for path in files:
        ids = [bos_id] + encode_without_model_limit(tokenizer, path.read_text(encoding="utf-8")) + [eos_id]
        document_lengths.append(len(ids))
        stream.extend(ids)
        while len(stream) >= sequence_length:
            chunk = stream[:sequence_length]
            packed_ids.append(chunk)
            packed_mask.append([1] * sequence_length)
            stream = stream[sequence_length:]
    remainder = len(stream)
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    pq.write_table(
        pa.table({"input_ids": packed_ids, "attention_mask": packed_mask}),
        parquet_path,
    )
    stats = {
        "tokenizer": model_id,
        "documents_packed": len(files),
        "total_tokens_in_stream": sum(document_lengths),
        "average_document_length_tokens": round(sum(document_lengths) / max(len(document_lengths), 1), 2),
        "packed_sequences": len(packed_ids),
        "sequence_length": sequence_length,
        "unused_remainder_tokens": remainder,
        "path": str(parquet_path),
    }
    del packed_ids, packed_mask, stream
    gc.collect()
    return tokenizer, stats

# Safely load splits if they are not in global memory
if "train_files" not in globals():
    train_files, eval_files = split_corpus_files(CLEAN_CORPUS)
    eval_texts = [p.read_text(encoding="utf-8") for p in eval_files]

tokenizer, packing_stats = tokenize_and_pack(train_files, PACKED_PARQUET, MODEL_ID, SEQUENCE_LENGTH)
print(json.dumps(packing_stats, indent=2))
(OUTPUTS / "packing_stats.json").write_text(json.dumps(packing_stats, indent=2), encoding="utf-8")
memory_snapshot("after_pack")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1480 > 1024). Running this sequence through the model will result in indexing errors


{
  "tokenizer": "microsoft/biogpt-large",
  "documents_packed": 123,
  "total_tokens_in_stream": 2772104,
  "average_document_length_tokens": 22537.43,
  "packed_sequences": 2707,
  "sequence_length": 1024,
  "unused_remainder_tokens": 136,
  "path": "/content/bits_llm_assignment_1_sem3/outputs/medical_packed_train.parquet"
}
{'tag': 'after_pack', 'cpu_total_gb': 12.67, 'cpu_available_gb': 6.48, 'cpu_process_gb': 1.83, 'gpu_allocated_gb': 3.3, 'gpu_reserved_gb': 6.46, 'gpu_free_gb': 7.97, 'gpu_total_gb': 14.56}


{'tag': 'after_pack',
 'cpu_total_gb': 12.67,
 'cpu_available_gb': 6.48,
 'cpu_process_gb': 1.83,
 'gpu_allocated_gb': 3.3,
 'gpu_reserved_gb': 6.46,
 'gpu_free_gb': 7.97,
 'gpu_total_gb': 14.56}

## Part A, Step 3 — Load the model, inspect it, save a baseline (2 marks)

On CUDA, load BioGPT-Large **directly onto the GPU** — parking 1.5B weights in the 15 GB system RAM first is a common Colab crash. On CPU/MPS the same helper loads fp32 into host (or unified) memory instead. After the three domain prompts are saved, the model is deleted so CPT can load a fresh copy.


In [37]:
def generate_outputs(model, tokenizer, prompts, max_new_tokens=60):
    import torch
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    device = next(model.parameters()).device
    results = []
    for prompt in prompts:
        inputs = {k: v.to(device) for k, v in tokenizer(prompt, return_tensors="pt").items()}
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        results.append({"prompt": prompt, "generated_text": tokenizer.decode(generated[0], skip_special_tokens=True)})
    return results


from transformers import AutoTokenizer

disable_optional_scipy_backend()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if not BUDGET.get("can_load_model", True):
    print("Skipping model load:", BUDGET["reason"])
else:
    base_model = load_causal_lm(MODEL_ID, MODEL_REVISION, four_bit=False, gradient_checkpointing=False, runtime=RUNTIME)
    audit = {"model_id": MODEL_ID, "device": RUNTIME["device"], "dtype": RUNTIME["dtype"], **model_audit(base_model)}
    print(json.dumps(audit, indent=2))
    (OUTPUTS / "model_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")

    baseline = generate_outputs(base_model, tokenizer, DOMAIN_PROMPTS)
    (OUTPUTS / "baseline_outputs.json").write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    for row in baseline:
        print("---")
        print(row["prompt"])
        print(row["generated_text"][:400])

    release_model_memory(base_model)
    base_model = None
memory_snapshot("after_baseline")


Loading weights:   0%|          | 0/772 [00:00<?, ?it/s]

{
  "model_id": "microsoft/biogpt-large",
  "device": "cuda",
  "dtype": "bfloat16",
  "total_parameters": 1571188800,
  "trainable_parameters": 1571188800,
  "decoder_layers": 48,
  "attention_heads": 25,
  "hidden_size": 1600,
  "head_dimension": 64,
  "vocab_size": 57717,
  "lm_head_output_dimension": 57717,
  "lm_head_matches_vocab": true
}
---
Explain why repeated blood-pressure measurements are useful in hypertension.
Explain why repeated blood-pressure measurements are useful in hypertension. < / FREETEXT > < / TITLE > ▃
---
What is the relationship between insulin resistance and type 2 diabetes?
What is the relationship between insulin resistance and type 2 diabetes? < / FREETEXT > < / TITLE > ▃ < ABSTRACT > < FREETEXT > Insulin resistance is a common feature of type 2 diabetes and is also associated with a number of other conditions, including obesity, hypertension, dyslipidemia, and polycystic ovary syndrome. Insulin resistance is also associated with an increased risk of car

{'tag': 'after_baseline',
 'cpu_total_gb': 12.67,
 'cpu_available_gb': 6.48,
 'cpu_process_gb': 1.73,
 'gpu_allocated_gb': 3.3,
 'gpu_reserved_gb': 6.41,
 'gpu_free_gb': 8.01,
 'gpu_total_gb': 14.56}

## Part A, Step 4 — Continual pre-training (2 marks)

Hugging Face `Trainer` + AdamW (8-bit on T4) + linear warmup. A custom callback records loss. Starting loss should be ~2–4; ~10.8 means the weights were random.

**How many steps?** `src/runtime.py` picks this from the machine:
- Colab T4: **60 steps** (~1 hour at the measured ~57 s/step), 8-bit Adam, full-weight CPT.
- A100/L40S: one packed epoch (~350 steps), full AdamW.
- **CPU / Apple MPS:** full AdamW on 1.5B fp32 does not fit (~6 GB weights + ~6 GB grads + ~12 GB Adam). CPT therefore uses **LoRA** (`q_proj`/`v_proj`, r=8), sequence length **256**, batch 1, **8 steps** (16 on MPS), then merges the adapter back into the base weights so the saved `biogpt-large-cpt/` folder is a normal model. Slow, but Run all completes without a GPU.


In [38]:
import torch
import gc
import sys
from transformers import AutoTokenizer

# 1. Delete explicit model and trainer objects from global namespace while preserving tokenizer
for var_name in ["cpt_model", "base_model", "trainer", "model"]:
    if var_name in globals():
        print(f"Deleting lingering global variable: {var_name}")
        del globals()[var_name]

# Ensure tokenizer exists and is loaded
if "tokenizer" not in globals():
    print("Reinitializing tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
else:
    print("Tokenizer is preserved in global namespace.")

# 2. Clear IPython history/output caches which hold hidden references to returned cell values
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('_', None)
        ipython.user_ns.pop('__', None)
        ipython.user_ns.pop('___', None)
        # Flush output history
        ipython.user_ns['_ih'] = []
        ipython.user_ns['_oh'] = {}
        ipython.user_ns['_dh'] = []
        print("IPython output caches cleared.")
except Exception as e:
    print(f"Could not clear IPython cache: {e}")

# 3. Force garbage collection multiple times to break circular references
for _ in range(5):
    gc.collect()

# 4. Empty PyTorch's internal CUDA allocator cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("CUDA memory cleaned up successfully while preserving tokenizer.")
from src.runtime import memory_snapshot
memory_snapshot("after_explicit_cleanup")

Deleting lingering global variable: cpt_model
Deleting lingering global variable: base_model
Tokenizer is preserved in global namespace.
IPython output caches cleared.
CUDA memory cleaned up successfully while preserving tokenizer.
{'tag': 'after_explicit_cleanup', 'cpu_total_gb': 12.67, 'cpu_available_gb': 6.48, 'cpu_process_gb': 1.73, 'gpu_allocated_gb': 3.3, 'gpu_reserved_gb': 3.64, 'gpu_free_gb': 10.79, 'gpu_total_gb': 14.56}


{'tag': 'after_explicit_cleanup',
 'cpu_total_gb': 12.67,
 'cpu_available_gb': 6.48,
 'cpu_process_gb': 1.73,
 'gpu_allocated_gb': 3.3,
 'gpu_reserved_gb': 3.64,
 'gpu_free_gb': 10.79,
 'gpu_total_gb': 14.56}

In [ ]:
class PackedTextDataset:
    # Memory-map Parquet so 2844 sequences are not copied into a Python list.

    def __new__(cls, parquet_path):
        import torch
        import pyarrow.parquet as pq

        table = pq.read_table(parquet_path, memory_map=True)
        ids_col = table.column("input_ids")
        mask_col = table.column("attention_mask")

        class Dataset(torch.utils.data.Dataset):
            def __len__(self):
                return len(ids_col)

            def __getitem__(self, index):
                ids = torch.tensor(ids_col[index].as_py(), dtype=torch.long)
                mask = torch.tensor(mask_col[index].as_py(), dtype=torch.long)
                return {"input_ids": ids, "attention_mask": mask, "labels": ids.clone()}

        return Dataset()


def train_cpt(model, tokenizer, dataset, output_dir, max_steps, learning_rate=5e-5):
    from transformers import Trainer, TrainerCallback, TrainingArguments

    class LossHistoryCallback(TrainerCallback):
        def __init__(self):
            self.history = []

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs:
                self.history.append({"step": int(state.global_step), "loss": float(logs["loss"])})

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    callback = LossHistoryCallback()
    mem_kwargs = trainer_memory_kwargs(RUNTIME, BUDGET)

    # 8-bit Adam is required on a T4 so optimiser states fit. Faster GPUs use
    # full AdamW. CPU/MPS keep adamw_torch on the LoRA params only.
    opt_val = mem_kwargs.get("optim", "adamw_torch")
    if BUDGET.get("eight_bit_adam"):
        try:
            import bitsandbytes  # noqa: F401
            opt_val = "adamw_bnb_8bit"
        except ImportError:
            warnings.warn("bitsandbytes not found; using adamw_torch. CPT may OOM on a 15 GB T4.")
            opt_val = "adamw_torch"

    # Map warmup_ratio to explicit warmup_steps for compatibility
    ratio = BUDGET.get("warmup_ratio", 0.1)
    computed_warmup_steps = int(max_steps * ratio)

    args_kwargs = dict(
        output_dir=str(output_dir),
        max_steps=max_steps,
        learning_rate=learning_rate,
        per_device_train_batch_size=BUDGET.get("per_device_train_batch_size", 1),
        gradient_accumulation_steps=BUDGET.get("gradient_accumulation_steps", 8),
        warmup_steps=computed_warmup_steps,
        weight_decay=mem_kwargs.get("weight_decay", 0.01),
        optim=opt_val,
        fp16=mem_kwargs.get("fp16", False),
        bf16=mem_kwargs.get("bf16", False),
        gradient_checkpointing=True,
        logging_steps=mem_kwargs.get("logging_steps", 10),
        save_strategy=mem_kwargs.get("save_strategy", "no"),
        report_to=mem_kwargs.get("report_to", "none"),
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
    )
    if mem_kwargs.get("use_cpu"):
        args_kwargs["use_cpu"] = True
    try:
        args = TrainingArguments(**args_kwargs)
    except TypeError:
        args_kwargs.pop("use_cpu", None)
        args = TrainingArguments(**args_kwargs)

    trainer = Trainer(model=model, args=args, train_dataset=dataset, callbacks=[callback])
    trainer.train()
    # CPU LoRA-CPT is merged back so later cells load a normal causal LM folder.
    trainer.model = merge_lora_if_needed(trainer.model)
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    history = callback.history
    del trainer
    return history


def plot_loss_history(history, output_path):
    import matplotlib.pyplot as plt
    if not history:
        warnings.warn("No loss records; skip plot.")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([row["step"] for row in history], [row["loss"] for row in history], marker="o", markersize=3)
    ax.set_title("CPT training loss")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=140)
    plt.show()
    plt.close(fig)


# Recover packing_stats from saved JSON if missing in current session namespace
if "packing_stats" not in globals():
    stats_path = OUTPUTS / "packing_stats.json"
    if stats_path.exists():
        packing_stats = json.loads(stats_path.read_text(encoding="utf-8"))
    else:
        raise NameError("packing_stats is not defined in memory, and packing_stats.json is missing.")

n_sequences = packing_stats["packed_sequences"]
cpt_steps = cpt_steps_for_dataset(n_sequences, BUDGET)
print({"packed_sequences": n_sequences, "cpt_steps": cpt_steps, "gpu_tier": BUDGET.get("gpu_tier"), "why": BUDGET["reason"]})

if cpt_steps <= 0:
    print("Not enough memory to train BioGPT-Large on this machine. Data prep is done; CPT/SFT need ~10 GB RAM or a GPU.")
else:
    cpt_model = load_causal_lm(MODEL_ID, MODEL_REVISION, four_bit=False, gradient_checkpointing=True, runtime=RUNTIME)
    cpt_model = wrap_lora_for_cpt(cpt_model, BUDGET)
    history = train_cpt(cpt_model, tokenizer, PackedTextDataset(PACKED_PARQUET), CPT_DIR, max_steps=cpt_steps)
    (OUTPUTS / "cpt_loss_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    plot_loss_history(history, OUTPUTS / "cpt_loss_curve.png")
    print({
        "initial_logged_loss": history[0]["loss"] if history else None,
        "final_logged_loss": history[-1]["loss"] if history else None,
        "steps": cpt_steps,
        "checkpoint": str(CPT_DIR),
    })
    release_model_memory(cpt_model)
    cpt_model = None
memory_snapshot("after_cpt")

{'packed_sequences': 2707, 'cpt_steps': 300, 'why': 'Colab T4-class box (~15 GB GPU / ~15 GB RAM): 8-bit Adam, 4-bit QLoRA, load straight to GPU, one model at a time'}


Loading weights:   0%|          | 0/772 [00:00<?, ?it/s]

Step,Training Loss
10,2.884503


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Part A, Step 5 — Domain perplexity and catastrophic forgetting (2 marks)

Lower domain PPL after CPT means the model learned this literature. The same three general prompts check that it did not forget simple world knowledge. Base and CPT are loaded **one after the other**, never together.


In [ ]:
def perplexity(model, tokenizer, texts, max_length=1024):
    import torch
    model.eval()
    total_nll, total_tokens = 0.0, 0
    device = next(model.parameters()).device
    for text in texts:
        ids = encode_without_model_limit(tokenizer, text)
        for start in range(0, len(ids), max_length):
            chunk = ids[start:start + max_length]
            if len(chunk) < 2:
                continue
            input_ids = torch.tensor([chunk], dtype=torch.long, device=device)
            with torch.no_grad():
                logits = model(input_ids=input_ids).logits[:, :-1, :]
            targets = input_ids[:, 1:]
            loss_sum = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction="sum"
            )
            total_nll += float(loss_sum.item())
            total_tokens += targets.numel()
            del logits, input_ids, targets, loss_sum
    mean_nll = total_nll / max(total_tokens, 1)
    return {"mean_nll": mean_nll, "perplexity": math.exp(min(mean_nll, 20)), "tokens": total_tokens}


def forgetting_verdict(base_text, cpt_text, expected_terms):
    base_hits = sum(term.lower() in base_text.lower() for term in expected_terms)
    cpt_hits = sum(term.lower() in cpt_text.lower() for term in expected_terms)
    return "Degraded" if base_hits > 0 and cpt_hits == 0 else "Retained"


if not (CPT_DIR / "config.json").exists():
    print("CPT checkpoint not found; skipping evaluation. Run Step 4 first.")
else:
    base_model = load_causal_lm(MODEL_ID, MODEL_REVISION, four_bit=False, gradient_checkpointing=False, runtime=RUNTIME)
    base_ppl = perplexity(base_model, tokenizer, eval_texts, SEQUENCE_LENGTH)
    base_general = generate_outputs(base_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    release_model_memory(base_model)
    base_model = None
    memory_snapshot("after_base_ppl")

    cpt_model = load_causal_lm(str(CPT_DIR), revision=None, four_bit=False, gradient_checkpointing=False, runtime=RUNTIME)
    cpt_ppl = perplexity(cpt_model, tokenizer, eval_texts, SEQUENCE_LENGTH)
    cpt_general = generate_outputs(cpt_model, tokenizer, GENERAL_PROMPTS, max_new_tokens=30)
    release_model_memory(cpt_model)
    cpt_model = None

    reduction = 100 * (base_ppl["perplexity"] - cpt_ppl["perplexity"]) / max(base_ppl["perplexity"], 1e-12)
    comparison = []
    for i, prompt in enumerate(GENERAL_PROMPTS):
        comparison.append({
            "prompt": prompt,
            "base_output": base_general[i]["generated_text"],
            "cpt_output": cpt_general[i]["generated_text"],
            "verdict": forgetting_verdict(base_general[i]["generated_text"], cpt_general[i]["generated_text"], EXPECTED_TERMS[i]),
        })
    results = {
        "base_perplexity": base_ppl,
        "cpt_perplexity": cpt_ppl,
        "ppl_reduction_percent": round(reduction, 2),
        "general_domain_comparison": comparison,
    }
    (OUTPUTS / "evaluation_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    print(json.dumps(results, indent=2))
memory_snapshot("after_eval")


## Part B1 — Instruction dataset (2 marks)

Heuristic pairs taken only from the cleaned corpus (no extra LLM). At least 100 rows, 80/20 split, JSONL.


In [ ]:
def _normalize(text):
    return re.sub(r"\s+", " ", text).strip()


def _sentences(text):
    chunks = re.split(r"(?<=[.!?])\s+", _normalize(text))
    return [c.strip() for c in chunks if len(c.strip()) > 20]


def build_instruction_pairs(docs, max_pairs=120, min_chars=80):
    templates = [
        "Summarize this clinical point from domain text: {topic}",
        "Explain this medical statement in clear terms: {topic}",
        "What does the following domain evidence imply? {topic}",
        "Provide a concise evidence-based explanation: {topic}",
        "Based only on the corpus, clarify: {topic}",
    ]
    pairs, seen = [], set()
    for file_name, text in docs:
        for sentence in _sentences(text):
            if len(sentence) < min_chars:
                continue
            topic = sentence[:120].strip()
            instruction = templates[len(pairs) % len(templates)].format(topic=topic)
            response = sentence if len(sentence) <= 450 else sentence[:450].rsplit(" ", 1)[0] + "..."
            key = (instruction.lower(), response.lower())
            if key in seen:
                continue
            seen.add(key)
            pairs.append({"instruction": instruction, "response": response, "source_file": file_name})
            if len(pairs) >= max_pairs:
                return pairs
    return pairs


def write_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps({"instruction": row["instruction"], "response": row["response"]}, ensure_ascii=False) + "\n")


docs = [(p.name, p.read_text(encoding="utf-8", errors="ignore")) for p in sorted(CLEAN_CORPUS.glob("*.txt"))]
pairs = build_instruction_pairs(docs, max_pairs=120)
if len(pairs) < 100:
    raise ValueError(f"Only {len(pairs)} instruction pairs; need ≥100.")
rng = random.Random(42)
shuffled = pairs.copy()
rng.shuffle(shuffled)
split_at = max(1, min(len(shuffled) - 1, int(round(len(shuffled) * 0.8))))
train_rows, eval_rows = shuffled[:split_at], shuffled[split_at:]
write_jsonl(pairs, OUTPUTS / "instruction_dataset.jsonl")
write_jsonl(train_rows, OUTPUTS / "instruction_train.jsonl")
write_jsonl(eval_rows, OUTPUTS / "instruction_eval.jsonl")
b1 = {
    "documents_used": len(docs),
    "total_pairs": len(pairs),
    "train_pairs": len(train_rows),
    "eval_pairs": len(eval_rows),
    "method": "heuristic sentences from cleaned corpus",
    "synthetic_prompt_template": (
        "Read the text below and generate 10 instruction-response pairs in JSON format based ONLY on this text. "
        "Each entry must have instruction and response keys."
    ),
}
print(json.dumps(b1, indent=2))
(OUTPUTS / "dataset_counts.json").write_text(json.dumps(b1, indent=2), encoding="utf-8")


## Part B2 — Three QLoRA adapters (2 marks)

The brief asks for `q_proj` / `v_proj` / `o_proj`. BioGPT names the last of those **`out_proj`**, so adapter C trains `out_proj`. Each adapter is trained, evaluated on the three domain prompts, then **unloaded** before the next one starts. On CUDA, 4-bit NF4 keeps the 1.5B backbone inside 15 GB. On CPU/MPS there is no bitsandbytes, so the same LoRA adapters train on the fp32 backbone (only LoRA params have gradients).


In [ ]:
@dataclass(frozen=True)
class AdapterSpec:
    name: str
    rank: int
    alpha: int
    target_modules: tuple[str, ...]


ADAPTER_SPECS = [
    AdapterSpec("A", 8, 16, ("q_proj", "v_proj")),
    AdapterSpec("B", 16, 32, ("q_proj", "v_proj")),
    AdapterSpec("C", 32, 32, ("q_proj", "v_proj", "o_proj")),  # o_proj → out_proj on BioGPT
]


def _format_for_sft(tokenizer, instruction, response):
    messages = [{"role": "user", "content": instruction}, {"role": "assistant", "content": response}]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        except Exception:
            pass
    return f"User: {instruction}\nAssistant: {response}"


def train_qlora_adapter(spec: AdapterSpec, train_rows, eval_rows, output_dir, max_steps, prompts):
    import inspect
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import AutoTokenizer, TrainingArguments
    from trl import SFTTrainer
    try:
        from trl import SFTConfig
    except ImportError:
        SFTConfig = None

    sft_source = str(CPT_DIR) if (CPT_DIR / "config.json").exists() else MODEL_ID
    revision = None if sft_source == str(CPT_DIR) else MODEL_REVISION
    tok_src = sft_source if (Path(sft_source) / "tokenizer_config.json").exists() else MODEL_ID
    tok_kwargs = {} if Path(tok_src).exists() else {"revision": MODEL_REVISION}
    tokenizer = AutoTokenizer.from_pretrained(tok_src, **tok_kwargs)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    use_4bit = bool(BUDGET.get("four_bit")) and bool(RUNTIME.get("cuda"))
    model = load_causal_lm(sft_source, revision, four_bit=use_4bit, gradient_checkpointing=True, runtime=RUNTIME)
    if use_4bit:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model)
    targets = resolve_lora_targets(model, list(spec.target_modules))
    print(f"Adapter {spec.name}: LoRA r={spec.rank} α={spec.alpha} targets={targets}")
    model = get_peft_model(
        model,
        LoraConfig(r=spec.rank, lora_alpha=spec.alpha, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=targets),
    )

    train_ds = Dataset.from_list([{"text": _format_for_sft(tokenizer, r["instruction"], r["response"])} for r in train_rows])
    eval_ds = Dataset.from_list([{"text": _format_for_sft(tokenizer, r["instruction"], r["response"])} for r in eval_rows])

    tkwargs = trainer_memory_kwargs(RUNTIME, BUDGET)
    tkwargs.update({"max_steps": max_steps, "learning_rate": 2e-4, "output_dir": str(output_dir)})
    # SFT only trains LoRA. 8-bit/paged Adam is CUDA+bitsandbytes; CPU uses adamw_torch.
    if BUDGET.get("eight_bit_adam") and RUNTIME.get("cuda"):
        tkwargs["optim"] = "paged_adamw_8bit"
    else:
        tkwargs["optim"] = BUDGET.get("optim") or "adamw_torch"
    args_cls = TrainingArguments
    if SFTConfig is not None:
        params = inspect.signature(SFTConfig).parameters
        if "dataset_text_field" in params or "max_length" in params:
            args_cls = SFTConfig
            if "dataset_text_field" in params:
                tkwargs["dataset_text_field"] = "text"
            if "max_length" in params:
                tkwargs["max_length"] = int(BUDGET.get("sft_max_length") or 512)
    # Shorter SFT sequences than CPT — instruction pairs are small and 1024 wastes VRAM.
    try:
        args = args_cls(**tkwargs)
    except TypeError:
        tkwargs.pop("use_cpu", None)
        args = args_cls(**tkwargs)
    trainer_kwargs = {"model": model, "args": args, "train_dataset": train_ds, "eval_dataset": eval_ds}
    tparams = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in tparams:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in tparams:
        trainer_kwargs["tokenizer"] = tokenizer
    if "dataset_text_field" in tparams:
        trainer_kwargs["dataset_text_field"] = "text"
    if "max_seq_length" in tparams:
        trainer_kwargs["max_seq_length"] = int(BUDGET.get("sft_max_length") or 512)

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    eval_out = {prompt: generate_outputs(model, tokenizer, [prompt], max_new_tokens=80)[0]["generated_text"] for prompt in prompts}
    release_model_memory(trainer, model, tokenizer, train_ds, eval_ds)
    return eval_out


sft_steps = BUDGET["sft_max_steps"]
eval_cache = {}
if sft_steps <= 0:
    print("Not enough memory to train LoRA adapters on this machine. Part B1 JSONL files are already written.")
else:
    ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)
    trained = {}
    for spec in ADAPTER_SPECS:
        out_dir = ADAPTER_ROOT / f"adapter_{spec.name}"
        print(f"Training adapter {spec.name} for {sft_steps} steps...")
        eval_cache[spec.name] = train_qlora_adapter(spec, train_rows, eval_rows, out_dir, sft_steps, DOMAIN_PROMPTS)
        trained[spec.name] = str(out_dir)
        memory_snapshot(f"after_adapter_{spec.name}")
    (OUTPUTS / "trained_adapters.json").write_text(json.dumps(trained, indent=2), encoding="utf-8")
    (OUTPUTS / "b2_evaluation_cache.json").write_text(json.dumps(eval_cache, indent=2), encoding="utf-8")
    print("B2 complete:", trained)


## Part B3 — Compare adapters to the Part A baseline (1 mark)

Uses the generations cached in B2, so this cell does **not** load the 1.5B model again.


In [ ]:
import csv

baseline_path = OUTPUTS / "baseline_outputs.json"
cache_path = OUTPUTS / "b2_evaluation_cache.json"
if not baseline_path.exists():
    print("Baseline outputs missing; run Step 3.")
else:
    baseline_rows = json.loads(baseline_path.read_text(encoding="utf-8"))
    baseline_by_prompt = {row.get("prompt", ""): row.get("generated_text", "") for row in baseline_rows}
    evaluation_cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else eval_cache
    comparison = []
    for prompt in DOMAIN_PROMPTS:
        row = {"prompt": prompt, "baseline_output": baseline_by_prompt.get(prompt, "")}
        for spec in ADAPTER_SPECS:
            row[f"adapter_{spec.name}_output"] = evaluation_cache.get(spec.name, {}).get(prompt, "[adapter not trained — B2 was skipped]")
        comparison.append(row)
    with (OUTPUTS / "adapter_comparison.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
        writer.writeheader()
        writer.writerows(comparison)
    (OUTPUTS / "adapter_comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    try:
        import pandas as pd
        display(pd.DataFrame(comparison))
    except Exception:
        print(json.dumps(comparison, indent=2)[:4000])
    print("Best adapter is a judgement call: pick the one whose answers stay closest to the corpus and the question.")
memory_snapshot("done")


## Deliverables written under `outputs/`

| File | Step |
|------|------|
| `step1_cleaning_report.json` | A1 |
| `medical_packed_train.parquet` | A2 |
| `model_audit.json`, `baseline_outputs.json` | A3 |
| `biogpt-large-cpt/`, `cpt_loss_curve.png` | A4 |
| `evaluation_results.json` | A5 |
| `instruction_dataset.jsonl` (+ train/eval) | B1 |
| `qlora_adapters/adapter_{A,B,C}/` | B2 |
| `adapter_comparison.csv` | B3 |

On a Colab T4, CPT is about **60 steps (~1 hour)** at the measured ~57 s/step, then three short QLoRA runs. On A100/L40S, CPT is about one packed epoch. If there is no GPU, everything up through the JSONL dataset still runs.
